In [ ]:
from pathlib import Path
import hashlib
import json
import re
import pandas as pd
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
SEED = CONFIG['seed']
PREFIXES = {'arw4':'ARWFour', 'lognormal_grw':'GRW', 'ar1_s':'AROne',
            'hurdle_ar1_s_p':'HurdleAROne', 'ar6_6':'ARSix', 'se_hurdle_s':'SEHurdle'}
# maximum-likelihood career stages from Zhang et al., Scientific productivity as a random walk
STAGES = [(0,4,'StageOne'), (5,7,'StageTwo'), (8,13,'StageThree'), (14,20,'StageFour')]
FILES = {'annual':'curve_agreement.csv', 'restart':'reentry_agreement.csv', 'paths':'path_agreement.csv',
         'wasserstein':'annual_wasserstein_summary.csv', 'wasserstein_annual':'annual_wasserstein.csv'}

In [ ]:
annual = pd.read_csv(INPUT/FILES['annual']).set_index(['source','window','domain','scale','statistic'])
restart = pd.read_csv(INPUT/FILES['restart']).set_index('source')
paths = pd.read_csv(INPUT/FILES['paths']).set_index(['source','window','descriptor'])
w1 = pd.read_csv(INPUT/FILES['wasserstein']).set_index('source')

# annual output distances grouped by career year
distances = pd.read_csv(INPUT/FILES['wasserstein_annual'])
stage_rows = []
for start,end,stage in STAGES:
    selected = distances.loc[distances.age.between(start,end)]
    stage_rows.append(selected.groupby('source',sort=False).wasserstein1
                      .agg(mean_wasserstein1='mean',n_years='size').reset_index()
                      .assign(stage=stage,start_age=start,end_age=end))
stage_w1 = pd.concat(stage_rows,ignore_index=True)
stage_w1.to_csv(OUTPUT/'stage_wasserstein_summary.csv',index=False)
stage_w1 = stage_w1.set_index(['source','stage'])


In [ ]:
rows = []
PRECISION = {
    'SEHurdleAnnualWassersteinMean': 2,
    'SEHurdleAnnualWassersteinSlope': 3,
}
def add_macro(prefix, suffix, value, table, selection, column, decimals, scale=1):
    if suffix in ('AnnualWassersteinMean', 'AnnualWassersteinSlope'):
        decimals = 2
    if pd.notna(value):
        rows.append({'macro':prefix+suffix, 'table':table, 'selection':json.dumps(selection),
                     'statistic':column, 'raw_value':value, 'scale':scale,
                     'formatted_value':f'{value*scale:.{decimals}f}'})

for source,prefix in PREFIXES.items():
    for suffix,column in [('AnnualWassersteinMean','mean_wasserstein1'),
                          ('AnnualWassersteinSlope','slope_wasserstein1')]:
        add_macro(prefix,suffix,w1.loc[source,column],'wasserstein',{'source':source},column,4)
    for start,end,stage in STAGES:
        add_macro(prefix,'AnnualWasserstein'+stage+'Mean',stage_w1.loc[(source,stage),'mean_wasserstein1'],
                  'wasserstein_annual',{'source':source,'start_age':start,'end_age':end},'mean_wasserstein1',2)
    for statistic in ['mean','median','variance']:
        key = (source,'common_generated','annual','raw',statistic)
        fields = ('source','window','domain','scale','statistic')
        add_macro(prefix,'Annual'+statistic.title()+'NRMSEPercent',annual.loc[key,'nrmse'],
                  'annual',dict(zip(fields,key)),'nrmse',2,100)
    add_macro(prefix,'RestartHazardRMSEPoints',restart.loc[source,'risk_weighted_rmse'],
              'restart',{'source':source},'risk_weighted_rmse',2,100)
    key = (source,'common_generated','longest_observed_zero_run')
    add_macro(prefix,'LongestInactiveSpellKSD',paths.loc[key,'ks_D'],'paths',
              dict(zip(('source','window','descriptor'),key)),'ks_D',4)

catalog = pd.DataFrame(rows)
# only the model comparison table defines primary result tags
paper = (ROOT.parent/'manuscript/paper/report_main.tex').read_text()
tables = re.findall(r'\\begin\{table\*?\}.*?\\end\{table\*?\}', paper, flags=re.S)
comparison_table = next((table for table in tables if r'\label{tab:model-comparison}' in table), '')
primary_macros = set(re.findall(r'\\([A-Za-z]+)', comparison_table))
catalog['tag'] = catalog.macro.map(lambda name: name if name in primary_macros else 'si_'+name)

In [ ]:
provenance = pd.DataFrame([
    {'table':table, 'source_file':str((INPUT/file).resolve().relative_to(ROOT.parent)),
     'sha256':hashlib.sha256((INPUT/file).read_bytes()).hexdigest()} for table,file in FILES.items()
])
catalog.merge(provenance,on='table').to_csv(OUTPUT/'macro_manifest.csv',index=False)
lines = [
    '% generated by i_report; refresh with make report',
    '% annual raw curves and longest inactive spells use career years 6--20',
    '% annual Wasserstein mean and raw-scale OLS slope use career years 0--20',
    '% restart RMSE is risk-weighted and expressed in percentage points',
    '% Percent commands omit the percent sign'
]
for row in catalog.itertuples():
    value = row.formatted_value
    if row.tag != row.macro:
        lines.append(r'\expandafter\newcommand\csname '+row.tag+r'\endcsname{'+value+r'\xspace}')
        value = r'\csname '+row.tag+r'\endcsname'
    lines.append(r'\newcommand{'+chr(92)+row.macro+'}{'+value+r'\xspace}')
temporary = OUTPUT/'trajectory_results.tex.tmp'
temporary.write_text('\n'.join(lines)+'\n')
temporary.replace(OUTPUT/'trajectory_results.tex')
display(catalog[['tag','formatted_value']])

# one row per reported statistic, one column per primary model or observed reference
labels = {'empirical_complete':'dblp', 'lognormal_grw':'GRW', 'ar1_s':'AR(1)',
          'hurdle_ar1_s_p':'H-AR(1)', 'se_hurdle_s':'SE-H'}
labels = {source:labels[source] for source in ['empirical_complete',*CONFIG['primary_plot_models']]}
prefixes = {'empirical_complete':'DBLP',**PREFIXES}
level = pd.read_csv(INPUT/'report/macro_manifest.csv').query("alias_of.isna()")
trajectory = catalog.assign(source=catalog.selection.map(lambda selection: json.loads(selection)['source']))
comparison = pd.concat([level,trajectory],ignore_index=True).loc[:,['macro','source','raw_value','scale']]
comparison = comparison.loc[comparison.source.isin(labels)].copy()
comparison['result'] = [macro.removeprefix(prefixes[source]) for macro,source in
                        zip(comparison.macro,comparison.source)]
comparison['value'] = comparison.raw_value * comparison.scale
comparison['model'] = comparison.source.map(labels)
wide = comparison.pivot(index='result',columns='model',values='value')
wide = wide.reindex(index=comparison.result.drop_duplicates(),columns=labels.values())
wide.to_csv(OUTPUT/'primary_model_comparison.csv')
